# ART Interactive Quickstart (TCM)

This notebook shows how to:

1. Verify that the local ART backend is running.
2. Explore a small Traditional Chinese Medicine (TCM) dataset.
3. Convert real records into ART `TrajectoryGroup` objects.
4. Register and train a `TrainableModel` against the running backend.

> **Prereq:** In a terminal run `uv run art run --host 0.0.0.0 --port 7999` so the FastAPI server is ready before you execute the cells below.

In [1]:
import asyncio
import json
import os
from pathlib import Path

import httpx
import ipywidgets as widgets
import pandas as pd

from art.backend import Backend
from art.model import TrainableModel
from art.trajectories import Trajectory, TrajectoryGroup
from art.types import TrainConfig

## 1. Connect to the ART backend
Set the base URL (defaults to `http://localhost:7999`) and make sure `/healthcheck` returns `{"status": "ok"}`.

In [2]:
BASE_URL = os.environ.get("ART_BASE_URL", "http://localhost:7999")
health = httpx.get(f"{BASE_URL}/healthcheck", timeout=5)
health.raise_for_status()
health.json()

{'status': 'ok'}

## 2. Load a sample of real TCM records

For GRPO training, we need **multiple responses to the same prompt** with different rewards. This allows the algorithm to learn which responses are better.

Below we create a sample where one patient condition has multiple possible responses ranked by quality.

In [3]:
# For GRPO, we need the SAME prompt with DIFFERENT responses and rewards
# This creates a contrastive learning signal

TCM_SAMPLE = [
    # Same condition, different quality responses
    {
        "plant": "Astragalus membranaceus",
        "condition": "Fatigue with recurrent colds",
        "compounds": ["astragaloside IV", "polysaccharides"],
        "response": (
            "Combine 9 g Astragalus with 6 g Codonopsis and 3 g honey-fried Licorice in a 500 mL decoction. "
            "Instruct the patient to drink 150 mL twice daily for 10 days. Provide diet advice to avoid raw foods. "
            "Monitor for signs of improvement in energy levels and cold frequency."
        ),
        "reward": 0.92,  # Best: complete with monitoring
    },
    {
        "plant": "Astragalus membranaceus",
        "condition": "Fatigue with recurrent colds",
        "compounds": ["astragaloside IV", "polysaccharides"],
        "response": (
            "Use Astragalus root, 9 grams daily as a decoction. "
            "Can be combined with other qi-tonifying herbs."
        ),
        "reward": 0.65,  # Medium: basic but lacks detail
    },
    {
        "plant": "Astragalus membranaceus",
        "condition": "Fatigue with recurrent colds",
        "compounds": ["astragaloside IV", "polysaccharides"],
        "response": ("Take Astragalus for energy."),
        "reward": 0.35,  # Poor: too vague, no dosage
    },
    {
        "plant": "Astragalus membranaceus",
        "condition": "Fatigue with recurrent colds",
        "compounds": ["astragaloside IV", "polysaccharides"],
        "response": (
            "Prepare 9 g Astragalus with 6 g Codonopsis in 500 mL water, simmer for 30 minutes. "
            "Take 150 mL morning and evening for 2 weeks. Avoid cold and raw foods during treatment."
        ),
        "reward": 0.78,  # Good: detailed but missing monitoring
    },
]

tcm_df = pd.DataFrame(TCM_SAMPLE)
tcm_df[["condition", "response", "reward"]]

,condition,response,reward
0,Fatigue with recurrent colds,Combine 9 g Astragalus with 6 g Codonopsis and...,0.92
1,Fatigue with recurrent colds,"Use Astragalus root, 9 grams daily as a decoct...",0.65
2,Fatigue with recurrent colds,Take Astragalus for energy.,0.35
3,Fatigue with recurrent colds,Prepare 9 g Astragalus with 6 g Codonopsis in ...,0.78


In [4]:
# View the reward distribution - GRPO needs variance here
print(
    "Reward spread:",
    max(r["reward"] for r in TCM_SAMPLE) - min(r["reward"] for r in TCM_SAMPLE),
)
print("\nResponses ranked by reward:")
for i, rec in enumerate(sorted(TCM_SAMPLE, key=lambda x: x["reward"], reverse=True), 1):
    print(f"{i}. reward={rec['reward']:.2f}: {rec['response'][:60]}...")

Reward spread: 0.5700000000000001

Responses ranked by reward:
1. reward=0.92: Combine 9 g Astragalus with 6 g Codonopsis and 3 g honey-fri...
2. reward=0.78: Prepare 9 g Astragalus with 6 g Codonopsis in 500 mL water, ...
3. reward=0.65: Use Astragalus root, 9 grams daily as a decoction. Can be co...
4. reward=0.35: Take Astragalus for energy....


## 3. Convert records into ART trajectories
Each record becomes a short conversation (system → user → assistant) with a reward score and rich metadata. Adjust the helper if your production data stores additional fields.

In [5]:
SYSTEM_PROMPT = "You are a cautious Traditional Chinese Medicine assistant."


def build_messages(record: dict) -> list[dict]:
    user_prompt = (
        f"Patient condition: {record['condition']}\n"
        f"Highlighted materia medica: {record['plant']} (compounds: {', '.join(record['compounds'])}).\n"
        "Recommend a short plan that includes dosage, preparation tips, and monitoring guidance."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": record["response"]},
    ]


def build_trajectory(record: dict) -> Trajectory:
    return Trajectory(
        messages_and_choices=build_messages(record),
        reward=record["reward"],
        metadata={
            "plant": record["plant"],
            "condition": record["condition"],
            "compounds": ",".join(record["compounds"]),
        },
    ).finish()


trajectory_group = TrajectoryGroup([build_trajectory(rec) for rec in TCM_SAMPLE])
len(trajectory_group.trajectories)

4

## 4. Register a `TrainableModel`
Fill in the model name/project/base checkpoint that matches your environment. Registration prepares the backend and returns an OpenAI-compatible endpoint for inference.

In [6]:
MODEL_NAME = os.environ.get("ART_MODEL_NAME", "tcm-formulator")
PROJECT_NAME = os.environ.get("ART_PROJECT", "tcm-demo")
BASE_MODEL = os.environ.get("ART_BASE_MODEL", "mistralai/Mistral-7B-Instruct-v0.2")

backend = Backend(base_url=BASE_URL)
trainable_model = TrainableModel(
    name=MODEL_NAME,
    project=PROJECT_NAME,
    base_model=BASE_MODEL,
)

await trainable_model.register(backend)
trainable_model

TrainableModel(name='tcm-formulator', project='tcm-demo', entity=None, id=None, config=None, trainable=True, inference_api_key='default', inference_base_url='http://0.0.0.0:8000/v1', inference_model_name='tcm-formulator', base_model='mistralai/Mistral-7B-Instruct-v0.2')

### Accessing the inference endpoint

The `trainable_model` object has all the connection info:
- `.inference_base_url` → `http://0.0.0.0:8000/v1`
- `.inference_api_key` → `"default"`
- `.inference_model_name` → `"tcm-formulator"`

**Example usage:**

```python
from openai import OpenAI

client = OpenAI(
    base_url=trainable_model.inference_base_url,
    api_key=trainable_model.inference_api_key,
)

response = client.chat.completions.create(
    model=trainable_model.inference_model_name,
    messages=[{"role": "user", "content": "What herb helps with fatigue?"}]
)
print(response.choices[0].message.content)
```

**Via curl:**
```bash
curl http://localhost:8000/v1/models -H "Authorization: Bearer default"
```

## 5. Log validation data
Send the trajectories to the backend so you have a baseline evaluation split before training.

In [7]:
await trainable_model.log([trajectory_group], split="val")
"Logged validation batch"

'Logged validation batch'

## 6. Kick off GRPO training
Set conservative hyperparameters and launch a training step. This streams progress from the backend and writes checkpoints/wandb logs just like the rest of ART. Run this cell only when your GPU is ready.

In [8]:
learning_rate = float(os.environ.get("ART_LR", 5e-6))
beta = float(os.environ.get("ART_BETA", 0.0))
train_config = TrainConfig(learning_rate=learning_rate, beta=beta)

# allow_training_without_logprobs=True is needed when using plain dict messages
# instead of OpenAI Choice objects (which have logprobs from inference)
print(f"Training with lr={train_config.learning_rate} beta={train_config.beta}")
await trainable_model.train(
    [trajectory_group],
    config=train_config,
    _config={"allow_training_without_logprobs": True},
    verbose=True,
)
print("Training iteration completed")

Training with lr=5e-06 beta=0.0


train:   0%|          | 0/1 [00:00<?, ?it/s]

Training iteration completed


## 7. (Optional) Shut down the backend
Call this when you're done experimenting. It tells the FastAPI app to stop and cleans up the local vLLM instance.

In [9]:
# await backend.close()
"Call await backend.close() to stop the ART server"

'Call await backend.close() to stop the ART server'

## 8. Test the Trained Model

After training, test the model's responses to see improvement in quality, specificity, and safety guidance.

In [ ]:
from openai import OpenAI

# Connect to the vLLM inference server
client = OpenAI(
    base_url=trainable_model.inference_base_url,
    api_key=trainable_model.inference_api_key,
)

# Test prompt
test_prompt = """Patient has wind-cold pattern with headache, body aches, and clear nasal discharge. 
Recommend herbs with dosages and preparation."""

print("Testing trained model...")
print(f"Prompt: {test_prompt}\n")
print("-" * 60)

response = client.chat.completions.create(
    model=trainable_model.inference_model_name,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_prompt},
    ],
    max_tokens=300,
    temperature=0.7,
)

print("Response:")
print(response.choices[0].message.content)

### Alternative: Test via curl

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer default" \
  -d '{
    "model": "tcm-formulator",
    "messages": [{"role": "user", "content": "Patient has wind-cold pattern with headache, body aches, and clear nasal discharge. Recommend herbs with dosages and preparation."}]
  }' | jq -r '.choices[0].message.content'
```